In [1]:
from datetime import date

import polars as pl
import yaml
from common import read_day, daterange


def extract_horizontal_accuracy(days: list[date], raw_path: str) -> pl.DataFrame:
    acc = pl.DataFrame()
    for i in days:
        print(i.strftime("%Y-%m-%d"))
        data = read_day(i, raw_path)
        data = data.with_columns(pl.from_epoch(pl.col("timestamp"), time_unit="s"))

        acc = pl.concat(
            [
                acc,
                data.select(
                    pl.col.horizontal_accuracy,
                    pl.col.device_aid_type,
                    pl.col.location_method,
                ),
            ]
        )

        del data
    return acc


with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [2]:
try:
    ha_2023 = pl.read_parquet("../output/debug/ha_2023.parquet")
except FileNotFoundError:
    days = list(daterange(date(2023, 10, 1), date(2023, 12, 31)))
    ha_2023 = extract_horizontal_accuracy(days, config["dirs"]["raw_source"])
    ha_2023.write_parquet("../output/debug/ha_2023.parquet")

In [3]:
try:
    ha_2024 = pl.read_parquet("../output/debug/ha_2024.parquet")
except FileNotFoundError:
    # runs 6m
    days = list(daterange(date(2024, 1, 1), date(2024, 2, 29))) + list(
        daterange(date(2024, 10, 1), date(2024, 12, 31))
    )
    ha_2024 = extract_horizontal_accuracy(days, config["dirs"]["raw_source"])
    ha_2024.write_parquet("../output/debug/ha_2024.parquet")

In [4]:
try:
    ha_2025 = pl.read_parquet("../output/debug/ha_2025.parquet")
except FileNotFoundError:
    days = list(daterange(date(2025, 1, 1), date(2025, 12, 31)))
    ha_2025 = extract_horizontal_accuracy(days, config["dirs"]["raw_source"])
    ha_2025.write_parquet("../output/debug/ha_2025.parquet")

In [5]:
try:
    ha_stat_23 = pl.read_parquet("../output/debug/ha_stat_2023.parquet")
except FileNotFoundError:
    ha_stat_23 = (
        ha_2023.group_by(["location_method", "device_aid_type"])
        .agg(
            pl.col.horizontal_accuracy.mean().round(1).alias("mean"),
            pl.col.horizontal_accuracy.std().round(1).alias("std"),
            pl.col.horizontal_accuracy.min().alias("min"),
            pl.col.horizontal_accuracy.quantile(0.25).alias("25%"),
            pl.col.horizontal_accuracy.median().alias("50%"),
            pl.col.horizontal_accuracy.quantile(0.75).alias("75%"),
            pl.col.horizontal_accuracy.quantile(0.80).alias("80%"),
            pl.col.horizontal_accuracy.quantile(0.85).alias("85%"),
            pl.col.horizontal_accuracy.quantile(0.90).alias("90%"),
            pl.col.horizontal_accuracy.quantile(0.95).alias("95%"),
            pl.col.horizontal_accuracy.quantile(0.99).alias("99%"),
            pl.col.horizontal_accuracy.max().alias("max"),
            pl.col.horizontal_accuracy.count().alias("count"),
        )
        .sort(["location_method", "device_aid_type"])
    )
    ha_stat_23.write_parquet("../output/debug/ha_stat_2023.parquet")
ha_stat_23

location_method,device_aid_type,mean,std,min,25%,50%,75%,80%,85%,90%,95%,99%,max,count
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""cell""","""AAID""",378.7,500.7,13.8,20.5,75.85,560.6,986.0,986.0,1244.7,1244.7,1250.0,1250.0,12
"""fused""","""AAID""",36.9,152.4,4.0,10.1,14.5,20.0,21.0,23.9,31.1,49.9,797.0,3001.0,120791690
"""fused""","""IDFA""",37.4,147.0,4.0,10.2,14.5,20.0,21.2,24.1,32.1,53.4,798.0,3000.9,19892424
"""gps""","""AAID""",26.5,65.7,4.0,9.9,14.9,22.5,29.6,48.0,65.0,65.0,165.0,3000.8,166274686
"""gps""","""IDFA""",30.2,78.5,4.0,9.3,15.4,31.0,55.7,64.0,65.0,66.0,165.0,3001.0,57717589
"""wifi""","""AAID""",495.7,763.4,4.0,8.0,14.0,962.4,1215.8,1488.8,1745.9,2007.7,2828.3,3000.8,28651
"""wifi""","""IDFA""",401.3,692.7,4.0,8.0,12.0,698.2,1009.1,1287.9,1589.8,1892.5,2699.6,2998.9,6193


In [ ]:
print(
    ha_stat_23.drop(["80%", "85%"])
    .rename({"location_method": "location method", "device_aid_type": "aid type"})
    .to_pandas()
    .to_markdown(index=False)
)

| location_method   | aid_type   |   mean |   std |   min |   25% |   50% |   75% |    90% |    95% |    99% |    max |     count |
|:------------------|:-----------|-------:|------:|------:|------:|------:|------:|-------:|-------:|-------:|-------:|----------:|
| cell              | AAID       |  378.7 | 500.7 |  13.8 |  20.5 | 75.85 | 560.6 | 1244.7 | 1244.7 | 1250   | 1250   |        12 |
| fused             | AAID       |   36.9 | 152.4 |   4   |  10.1 | 14.5  |  20   |   31.1 |   49.9 |  797   | 3001   | 120791690 |
| fused             | IDFA       |   37.4 | 147   |   4   |  10.2 | 14.5  |  20   |   32.1 |   53.4 |  798   | 3000.9 |  19892424 |
| gps               | AAID       |   26.5 |  65.7 |   4   |   9.9 | 14.9  |  22.5 |   65   |   65   |  165   | 3000.8 | 166274686 |
| gps               | IDFA       |   30.2 |  78.5 |   4   |   9.3 | 15.4  |  31   |   65   |   66   |  165   | 3001   |  57717589 |
| wifi              | AAID       |  495.7 | 763.4 |   4   |   8   | 14    | 

In [8]:
try:
    ha_stat_24 = pl.read_parquet("../output/debug/ha_stat_2024.parquet")
except FileNotFoundError:
    ha_stat_24 = (
        ha_2024.group_by(["location_method", "device_aid_type"])
        .agg(
            pl.col.horizontal_accuracy.mean().round(1).alias("mean"),
            pl.col.horizontal_accuracy.std().round(1).alias("std"),
            pl.col.horizontal_accuracy.min().alias("min"),
            pl.col.horizontal_accuracy.quantile(0.25).alias("25%"),
            pl.col.horizontal_accuracy.median().alias("50%"),
            pl.col.horizontal_accuracy.quantile(0.75).alias("75%"),
            pl.col.horizontal_accuracy.quantile(0.80).alias("80%"),
            pl.col.horizontal_accuracy.quantile(0.85).alias("85%"),
            pl.col.horizontal_accuracy.quantile(0.90).alias("90%"),
            pl.col.horizontal_accuracy.quantile(0.95).alias("95%"),
            pl.col.horizontal_accuracy.quantile(0.99).alias("99%"),
            pl.col.horizontal_accuracy.max().alias("max"),
            pl.col.horizontal_accuracy.count().alias("count"),
        )
        .sort(["location_method", "device_aid_type"])
    )
    ha_stat_24.write_parquet("../output/debug/ha_stat_2024.parquet")
ha_stat_24

location_method,device_aid_type,mean,std,min,25%,50%,75%,80%,85%,90%,95%,99%,max,count
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""cell""","""AAID""",10.0,3.3,5.2,6.6,11.6,12.6,12.8,13.6,13.8,14.2,14.6,14.6,21
"""cell""","""IDFA""",12.0,3.5,5.2,9.9,12.5,15.0,15.1,15.6,16.0,16.2,16.3,16.3,38
"""fused""","""AAID""",37.5,156.6,4.0,10.2,14.5,20.0,21.0,23.9,31.2,49.9,799.0,3001.0,183387215
"""fused""","""IDFA""",34.4,137.4,4.0,10.0,14.4,20.0,21.0,23.6,30.9,48.9,700.0,3001.0,25983776
"""gps""","""AAID""",25.5,59.5,4.0,9.5,14.7,22.0,28.4,45.6,65.0,65.0,164.0,3000.9,278714910
"""gps""","""IDFA""",30.5,82.4,4.0,9.0,15.3,31.8,61.0,64.0,65.0,66.0,165.0,3001.0,85369524
"""wifi""","""AAID""",463.0,737.3,4.0,8.0,13.0,892.2,1152.0,1412.2,1678.1,1963.7,2776.3,3000.6,12667
"""wifi""","""IDFA""",440.0,704.4,4.0,8.0,13.0,862.2,1123.3,1347.5,1615.7,1907.4,2678.0,2985.8,2405


In [ ]:
print(
    ha_stat_24.drop(["80%", "85%"])
    .rename({"location_method": "location method", "device_aid_type": "aid type"})
    .to_pandas()
    .to_markdown(index=False)
)

| location method   | aid type   |   mean |   std |   min |   25% |   50% |   75% |    90% |    95% |    99% |    max |     count |
|:------------------|:-----------|-------:|------:|------:|------:|------:|------:|-------:|-------:|-------:|-------:|----------:|
| cell              | AAID       |   10   |   3.3 |   5.2 |   6.6 |  11.6 |  12.6 |   13.8 |   14.2 |   14.6 |   14.6 |        21 |
| cell              | IDFA       |   12   |   3.5 |   5.2 |   9.9 |  12.5 |  15   |   16   |   16.2 |   16.3 |   16.3 |        38 |
| fused             | AAID       |   37.5 | 156.6 |   4   |  10.2 |  14.5 |  20   |   31.2 |   49.9 |  799   | 3001   | 183387215 |
| fused             | IDFA       |   34.4 | 137.4 |   4   |  10   |  14.4 |  20   |   30.9 |   48.9 |  700   | 3001   |  25983776 |
| gps               | AAID       |   25.5 |  59.5 |   4   |   9.5 |  14.7 |  22   |   65   |   65   |  164   | 3000.9 | 278714910 |
| gps               | IDFA       |   30.5 |  82.4 |   4   |   9   |  15.3 | 

In [ ]:
# ha_stat_25 = (
#     ha_2025.group_by(["location_method", "device_aid_type"])
#     .agg(
#         pl.col.horizontal_accuracy.mean().round(1).alias("mean"),
#         pl.col.horizontal_accuracy.std().round(1).alias("std"),
#         pl.col.horizontal_accuracy.min().alias("min"),
#         pl.col.horizontal_accuracy.quantile(0.25).alias("25%"),
#         pl.col.horizontal_accuracy.median().alias("50%"),
#         pl.col.horizontal_accuracy.quantile(0.75).alias("75%"),
#         pl.col.horizontal_accuracy.quantile(0.80).alias("80%"),
#         pl.col.horizontal_accuracy.quantile(0.85).alias("85%"),
#         pl.col.horizontal_accuracy.quantile(0.90).alias("90%"),
#         pl.col.horizontal_accuracy.quantile(0.95).alias("95%"),
#         pl.col.horizontal_accuracy.quantile(0.99).alias("99%"),
#         pl.col.horizontal_accuracy.max().alias("max"),
#         pl.col.horizontal_accuracy.count().alias("count"),
#     )
#     .sort(["location_method", "device_aid_type"])
# )
# ha_stat_25.write_parquet("../output/debug/ha_stat_2025.parquet")
# ha_stat_25